In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
uploaded = files.upload()

Saving small_business_loans.csv to small_business_loans (1).csv


In [ ]:
# y = 1 if the borrower defaulted y = 0 if the loan was replayed
# late_payments: number of past late payments
# years_in_business: how long the business has operated

df = pd.read_csv("small_business_loans.csv")
df.head()

,late_payments,years_in_business,default
0,1,16,0
1,2,5,0
2,3,2,0
3,1,6,0
4,3,10,1


In [ ]:
#the outcome is either 0 or 1
df["default"].value_counts()

,count
default,
0,2278
1,722


**how to not write a model**

In [ ]:
#fitting a log regression in python
import statsmodels.formula.api as smf
# i want to predict the default based on late payments and the years in business
formula = "default ~ late_payments + years_in_business"
# i want the function to be a log function using the df and fit the model
model = smf.logit(formula=formula, data=df).fit()
print(model.summary())

# check for coefficients, therefore, formula = 1/ (1 + e^-(-0.96 + 0.52latepayments - 0.10yearsinbusiness))
# check for statistical significance, check if  P>|z| = 0, or less than 5%, if larger than 5% dont use
# we do NOT assess models with r^2

Optimization terminated successfully.
         Current function value: 0.506064
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                 3000
Model:                          Logit   Df Residuals:                     2997
Method:                           MLE   Df Model:                            2
Date:                Wed, 07 Oct 2026   Pseudo R-squ.:                 0.08296
Time:                        15:18:23   Log-Likelihood:                -1518.2
converged:                       True   LL-Null:                       -1655.5
Covariance Type:            nonrobust   LLR p-value:                 2.238e-60
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -0.9672      0.091    -10.585      0.000      -1.146      -0.788
late_pay

In [ ]:
#the fitted model gives the probability for any applicant (first applicant: 2 late payments, 5 years, second applicant: 7 late payments, 10 years in business)
applicant = pd.DataFrame({"late_payments":[2,7], "years_in_business":[5,10]})
print(model.predict(applicant))


0    0.391640
1    0.839928
dtype: float64


In [ ]:
#the model gives a probability, the decision is an action
#we pick a threshold, t

df["p_hat"] = model.predict(df) #predicted probability of default for every loan
t=0.2
total_predicted_defaults = (df["p_hat"] >= t).sum()
total_predicted_defaults #shows the amount of ppl that will NOT get a loan


np.int64(1806)

In [ ]:
df

,late_payments,years_in_business,default,p_hat
0,1,16,0,0.108120
1,2,5,0,0.391640
2,3,2,0,0.597697
3,1,6,0,0.255709
4,3,10,1,0.392341
...,...,...,...,...
2995,2,9,0,0.297944
2996,0,6,0,0.169077
2997,1,16,0,0.108120
2998,1,2,0,0.342604


In [ ]:
#lets try 0.5, and then calculate a confusion matrix
t=0.5
total_predicted_defaults = (df["p_hat"] >= t).sum()
total_predicted_defaults #shows the amount of ppl that will NOT get a loan


np.int64(111)

In [ ]:
# false negative (fn): approved a loan that defaults
# false positive (fp): reject a borrower that would have repaid
# true negative (tn) : approved a loan that doesnt default
# true positive (tp): reject a borrower that would have default

In [ ]:
# accuracy tp + tn/ total, how often are we right overall?
# precision tp / tp + fp, when we flag a loan, how often is it truly a default?
# recall tp / tp + fn,  of all real defaults, how many did we catch?
# specificity tn/tn+fp, of all good loans, how many did we correctly clear

In [ ]:
# calculate confusion matrix
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score
pred = (df["p_hat"] >= 0.5).astype(int)
cm = confusion_matrix(df["default"], pred) #scikit-learn order: [[TN, FP], [FN, TP]]
print(cm)

[[2241   37]
 [ 648   74]]


**how to actually write a model**

In [ ]:
from sklearn.model_selection import train_test_split
# step 1: split the data
train, test = train_test_split(
    df,
    test_size=0.25,          # 25% held back for testing
    stratify=df["default"],  # same default rate in both sets
    random_state=42)         # reproducible split

#step 2: fit on the training set only
formula = "default ~ late_payments + years_in_business"
model = smf.logit (formula, data=train).fit()

Optimization terminated successfully.
         Current function value: 0.506543
         Iterations 6


In [ ]:
from sklearn.metrics import confusion_matrix

#step 3: predicted probability of default for each test loan
p_test = model.predict(test)
#step 4: apply the threshold from the cost analysis
pred = (p_test >= 0.09). astype(int)
#step 5: compute confusion matrix on test set
print(confusion_matrix(test["default"], pred)) #scikit-learn order: [[TN, FP], [FN, TP]]

[[ 55 515]
 [  2 178]]


reduced risk/cost by reducing false negatives, approved loan that would have defaulted

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

print("accuracy   :", round(accuracy_score(test["default"], pred), 3))
print("precision  :", round(precision_score(test["default"], pred), 3))
print("recall     :", round(recall_score(test["default"], pred), 3))
print("specificity:", round(recall_score(test["default"], pred, pos_label=0), 3))

accuracy   : 0.311
precision  : 0.257
recall     : 0.989
specificity: 0.096
